# asyncio 事件循环与任务时间线

## 学习目标

通过可调时间线理解 coroutine、Task、事件循环和 `await` 的关系，并区分：

- Task 的创建顺序；
- I/O 等待完成顺序；
- `asyncio.gather` 的结果顺序；
- 取消请求与 `finally` 清理。

核心心智模型是：`await` 不会让等待本身变快，它让当前 Task 暂停，把事件循环交给其他
可以继续运行的 Task。

## 准备

Plotly 负责绘制可悬浮查看的等待时间线，`ipywidgets` 负责调整三个任务的模拟 I/O
延迟。时间线中的彩色条表示 Task 正在等待外部结果，不表示 CPU 一直在执行该任务。

In [ ]:
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

PLOTLY_CONFIG = {
    "displayModeBar": False,
    "displaylogo": False,
    "responsive": True,
}

## 操作任务时间线

调整任一延迟后观察两件事：

1. 三个 Task 都从时间零附近开始等待，说明它们已经并发启动；
2. 完成顺序由延迟决定，但 `gather` 仍按传入顺序返回结果。

In [ ]:
TASK_COLORS = {
    "任务 A": "#2563eb",
    "任务 B": "#f97316",
    "任务 C": "#16a34a",
}


def draw_task_timeline(delay_a, delay_b, delay_c):
    names = ["任务 A", "任务 B", "任务 C"]
    delays = [delay_a, delay_b, delay_c]
    indexed_tasks = list(enumerate(zip(names, delays, strict=True)))
    completed = sorted(indexed_tasks, key=lambda item: (item[1][1], item[0]))
    completion_order = [name for _, (name, _) in completed]

    figure = go.Figure()
    figure.add_trace(
        go.Bar(
            x=delays,
            y=names,
            orientation="h",
            marker_color=[TASK_COLORS[name] for name in names],
            text=[f"等待 {delay:.1f} 秒" for delay in delays],
            textposition="inside",
            hovertemplate="%{y}<br>等待：%{x:.1f} 秒<extra></extra>",
            name="等待外部 I/O",
        )
    )
    figure.add_trace(
        go.Scatter(
            x=delays,
            y=names,
            mode="markers+text",
            marker={"size": 14, "color": "#0f172a", "symbol": "diamond"},
            text=["恢复并完成"] * len(names),
            textposition="top center",
            hovertemplate="%{y}<br>在 %{x:.1f} 秒恢复<extra></extra>",
            name="恢复点",
        )
    )
    figure.update_layout(
        title="三个 Task 的模拟 I/O 等待与恢复",
        xaxis_title="相对时间（秒）",
        yaxis_title="",
        xaxis={"range": [0, max(delays) + 0.25]},
        yaxis={"autorange": "reversed"},
        height=390,
        margin={"l": 80, "r": 30, "t": 70, "b": 55},
        showlegend=True,
    )
    figure.show(config=PLOTLY_CONFIG)
    print("实际完成顺序：", " → ".join(completion_order))
    print("gather 返回顺序：任务 A → 任务 B → 任务 C")


delay_a = widgets.FloatSlider(
    value=0.6,
    min=0.1,
    max=1.0,
    step=0.1,
    description="任务 A：",
    continuous_update=False,
)
delay_b = widgets.FloatSlider(
    value=0.4,
    min=0.1,
    max=1.0,
    step=0.1,
    description="任务 B：",
    continuous_update=False,
)
delay_c = widgets.FloatSlider(
    value=0.2,
    min=0.1,
    max=1.0,
    step=0.1,
    description="任务 C：",
    continuous_update=False,
)
timeline_output = widgets.interactive_output(
    draw_task_timeline,
    {
        "delay_a": delay_a,
        "delay_b": delay_b,
        "delay_c": delay_c,
    },
)
display(
    widgets.HBox(
        [widgets.VBox([delay_a, delay_b, delay_c]), timeline_output]
    )
)

## 运行真实协程

上面的图使用确定的延迟构造概念时间线。下面让事件循环真正调度三个协程，并记录它们
的完成顺序。耗时会受机器调度影响，所以只观察相对顺序，不把具体毫秒数当成固定答案。

In [ ]:
import asyncio
import time


async def simulated_request(name, delay, completion_log):
    await asyncio.sleep(delay)
    completion_log.append(name)
    return f"{name}:结果"


async def run_concurrent_requests():
    completion_log = []
    started_at = time.perf_counter()
    results = await asyncio.gather(
        simulated_request("A", 0.06, completion_log),
        simulated_request("B", 0.04, completion_log),
        simulated_request("C", 0.02, completion_log),
    )
    elapsed = time.perf_counter() - started_at
    return completion_log, results, elapsed


completion_log, results, elapsed = await run_concurrent_requests()
print("完成顺序:", completion_log)
print("gather 结果:", results)
print(f"总耗时约 {elapsed:.3f} 秒，而不是三个延迟简单相加")

`C` 最先完成，是因为它等待时间最短；`gather` 的结果仍对应传入的 A、B、C。
这两个顺序服务不同目的：完成顺序描述调度事实，结果顺序让调用者容易对应输入。

## 取消不是强行终止

`task.cancel()` 会安排在合适的暂停点向 Task 抛出 `CancelledError`。协程仍有机会执行
`finally`，释放锁、连接或临时资源，然后取消继续向上传播。

In [ ]:
async def cancellable_worker(events):
    events.append("开始")
    try:
        await asyncio.sleep(10)
    finally:
        events.append("finally 清理")


async def cancellation_demo():
    events = []
    task = asyncio.create_task(cancellable_worker(events))
    await asyncio.sleep(0)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        events.append("调用者收到取消")
    return events


print("取消过程:", await cancellation_demo())

## 常见误解

- “asyncio 会让 CPU 计算并行”：事件循环主要提升大量 I/O 等待场景的利用率。
- “创建 coroutine 就开始运行”：coroutine 需要被 `await` 或包装成 Task 才会推进。
- “`gather` 的结果顺序就是完成顺序”：结果顺序默认与输入顺序一致。
- “取消可以忽略”：吞掉 `CancelledError` 会破坏超时、TaskGroup 和服务关闭流程。

## 面试时可以这样解释

> Task 是被事件循环调度的 coroutine。协程遇到尚未完成的 `await` 时保存状态并让出
> 控制权；等待完成后，事件循环再从暂停点恢复它。取消也是通过异常在暂停点协作传播，
> 因此清理逻辑应放在 `finally` 中。

## 继续探索

1. 把一个任务中的 `asyncio.sleep` 换成 `time.sleep`，观察其他任务为什么一起变慢。
2. 用 `TaskGroup` 重写示例，观察一个子任务失败时兄弟任务如何被取消。
3. 给时间线增加“创建、等待、恢复、完成”四种事件，而不是只画等待区间。